In [ ]:
import requests  # Librería que uso para hacer solicitudes HTTP a la API
import sqlite3   # Módulo para trabajar con bases de datos SQLite
import pandas as pd  # Librería para manipular y analizar datos en formato tabla

# Dirección del servicio API del que voy a obtener la información
API_URL = "https://jsonplaceholder.typicode.com/users"


# Función que se encarga de consultar la API
# Verifica que la respuesta sea exitosa (código 200)
# Si falla, muestra el código de error correspondiente
def extraer_datos(url):
    respuesta = requests.get(url)
    if respuesta.status_code == 200:
        return respuesta.json()
    else:
        raise Exception(f"No fue posible conectarse a la API. Código: {respuesta.status_code}")


# Función que guarda la información en una base de datos SQLite
# Crea la base de datos si no existe y define la tabla usuarios
def guardar_en_db(datos):
    conexion = sqlite3.connect('datos_api.db')
    cursor = conexion.cursor()

    # Creación de la tabla usuarios con sus respectivos campos
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            usuario TEXT,
            email TEXT
        )
    ''')

    # Recorrido de los datos obtenidos desde la API
    # Se insertan o actualizan en la tabla usuarios
    for elemento in datos:
        cursor.execute('''
            INSERT OR REPLACE INTO usuarios (id, nombre, usuario, email)
            VALUES (?, ?, ?, ?)
        ''', (elemento['id'], elemento['name'], elemento['username'], elemento['email']))

    # Se guardan los cambios realizados
    conexion.commit()

    # Se cierra la conexión con la base de datos
    conexion.close()


# Función encargada de generar archivos de evidencia (CSV y auditoría)
def generar_evidencias(datos_api):

    # Conexión a la base de datos para extraer los registros almacenados
    conexion = sqlite3.connect('datos_api.db')
    df = pd.read_sql_query("SELECT * FROM usuarios", conexion)

    # Exportación de los datos a un archivo CSV
    df.to_csv('muestra_usuarios.csv', index=False)
    conexion.close()

    # Cantidad de registros obtenidos directamente desde la API
    total_api = len(datos_api)

    # Cantidad de registros almacenados en la base de datos
    total_db = len(df)

    # Creación del archivo de auditoría en formato texto
    with open('auditoria.txt', 'w', encoding='utf-8') as archivo:
        archivo.write("INFORME DE AUDITORÍA\n")
        archivo.write("====================\n")
        archivo.write(f"Total de registros obtenidos desde la API: {total_api}\n")
        archivo.write(f"Total de registros almacenados en SQLite: {total_db}\n")

        # Validación para comprobar que los datos coincidan
        if total_api == total_db:
            archivo.write("\nRESULTADO: El proceso se realizó correctamente.\n")
        else:
            archivo.write("\nRESULTADO: Hay inconsistencias en los datos.\n")


# BLOQUE PRINCIPAL DE EJECUCIÓN
try:
    print("Iniciando solicitud de datos a la API...")
    datos = extraer_datos(API_URL)

    print("Guardando información en la base de datos datos_api.db...")
    guardar_en_db(datos)

    print("Creando archivos de respaldo (CSV y auditoría)...")
    generar_evidencias(datos)

    print("Proceso finalizado sin errores.")
except Exception as error:
    print(f"Se presentó un inconveniente durante la ejecución: {error}")